Updated code keeps crashing on TCgiSeeding for select years and ensemble members 

## times to try: 
- 2004 and realization 1
- 2009 and realization 3

In [1]:

#!/usr/bin/env python
import numpy as np

import os


import Namelist as gv
import pandas as pd
import netCDF4 as nc
from netCDF4 import Dataset
from scipy import stats
from tools.util import argminDatetime
from tools.util import int2str,date_interpolation
from scipy.io import loadmat, netcdf_file
from datetime import datetime,timedelta
from dask.diagnostics import ProgressBar

import time
import calendar



In [2]:
gv.overwrite

False

In [3]:
output_fname = f"{gv.output_path}{gv.Model}_2009_ens003.nc"
print(output_fname)
os.path.exists(output_fname)



output/ERA5_2009_ens003.nc


True

In [4]:

if not gv.overwrite and os.path.exists(output_fname):
    print('this passes')
    print(gv.overwrite)
    print(output_fname)

else:
    print('this runs')
    print(gv.overwrite)

this passes
False
output/ERA5_2009_ens003.nc


In [5]:
output_fname

'output/ERA5_2009_ens003.nc'

In [6]:
seed = gv.random_seed
rng = np.random.default_rng(seed=seed)

In [7]:

def TCgiSeeding (gxlon, gxlat, gi, climInitLon, climInitLat, climInitDate, ratio, iy):
    '''
    This is a subroutine for calculating tropical cyclone genesis locations based on the
    genesis index (gi).
    
    ---- Parameters -------------------------------------- 
    gxlon : 2D array (nlat x nlon)
        mesh-grid of longitudes

    gxlat : 2D array (nlat x nlon)
        mesh-grid of latitudes
    
    gi : 3D array (month x nlat x nlon)
        Monthly tropical cyclone genisis index (gi). 
        "gi is tropical cyclone genesis index. We use nansum(gi) to for
        total number of seeds, and then randomly select location using gi
        as genesis probability."
            [higher values == more likely genisis?] TODO 
            [spatial probability distribution of cyclone 
            formation for each month.]
    
    climInitLon, climInitLat, climInitDate: lists
        Lists storing storing previous seeds information (lon, lat, and date)

    ratio: float
        how many more seeds we should give [create?], based on the survival rate
        [Scaling factor determining how many seeds to create relative to the total
        genesis index sum (accounts for survival rate).]

    iy: int
        target year for seeding

    ---- Returns ---------------------------------------- 
    None (modifies the climInitLon, climInitLat, and climInitDate lists in place)

    '''

    # loop through year (iy) by month (im) 
    for im in range(12):
        xk, pk = np.arange(gi[im,:,:].size), gi[im,:,:].ravel()

        ####### UPDATING

        ## TODO: find way to fix when multiple runs x1 produces empty/full of nans 
        ## remove points where gi OR the coordinate grids are NaN
        # lon_flat = gxlon.ravel()
        # lat_flat = gxlat.ravel()
        # valid = np.isfinite(pk) & np.isfinite(lon_flat) & np.isfinite(lat_flat)
        # xk, pk = xk[valid], pk[valid]

        # custm = stats.rv_discrete(name='custm', values=(xk, pk/pk.sum()), seed=rng)

        # n_seeds = np.int_(np.rint(pk.sum()*ratio))
        # r = custm.rvs(size=n_seeds)

        # iix = gxlon.ravel()[r.ravel()]
        # iiy = gxlat.ravel()[r.ravel()]

        # iday = rng.choice(np.arange(calendar.monthrange(iy,im+1)[1]),
        #                    size=n_seeds, replace=True)

        # if iday.size > 0:
        #     for id in range(iday.size):

        #         lon0, lat0 = iix[id], iiy[id]

        #         ## belt-and-suspenders: skip if somehow still NaN/inf
        #         if not (np.isfinite(lon0) and np.isfinite(lat0)):
        #             continue

        #         ## linspace guarantees a fixed, non-empty length regardless
        #         ## of float rounding, unlike arange
        #         x1 = np.linspace(lon0-1, lon0+1, 201)
        #         y1 = np.linspace(lat0-1, lat0+1, 201)

        #         xx = rng.choice(x1, 1)
        #         yy = rng.choice(y1, 1)

        #         if gv.debugging: print(xx)
        #         if gv.debugging: print(yy)

        #         climInitDate.append(datetime(iy,im+1,iday[id]+1,0,0))
        #         climInitLon.append(xx)
        #         climInitLat.append(yy)

        ## OLD

        ### this maybe isn't working as well as it could 
        ## remove NaNs
        dummy = xk*pk
        xk, pk = xk[dummy==dummy], pk[dummy==dummy] # (dummy==dummy is False for NaNs)
        # Build discrete probability distribution (custm) using gi values as weights
        # xk = possible grid indices, pk/pk.sum() = normalized probabilities
        custm = stats.rv_discrete(name='custm', values=(xk, pk/pk.sum()), seed = rng)

        #print(custm)

        ## randomly select grid point from custm 
        ## number of points selected [np.rint(pk.sum()*ratio)]
            # proportional to sum of gensis probability * ratio
        r = custm.rvs(size=np.int_(np.rint(pk.sum()*ratio)))

        ## map selected grid points into lon/lat coordinated
        iix = gxlon.ravel()[r.ravel()]
        iiy = gxlat.ravel()[r.ravel()]
        ## randomly select day in month to assign each seed 
        ## update to use set rng
        iday = rng.choice(np.arange(calendar.monthrange(iy,im+1)[1]), size=np.int_(np.rint(pk.sum()*ratio)), replace=True)


        # if there are days assigned a seed in the month:
        if iday.size>0:
            for id in range(iday.size):

                ## this is occasionally generating empty arrays
                x1 = np.arange(iix[id]-1,iix[id]+1.01, 0.01)
                y1 = np.arange(iiy[id]-1,iiy[id]+1.01, 0.01)

                #print(x1)
                
                ## update to use set rng
                xx = rng.choice(x1, 1)
                yy = rng.choice(y1, 1)

                # if gv.debugging: print(xx)
                # if gv.debugging: print(yy)

                # append generated seed information to appropriate lists
                climInitDate.append(datetime(iy,im+1,iday[id]+1,0,0))
                climInitLon.append(xx)
                climInitLat.append(yy)



    return climInitLon, climInitLat, climInitDate

In [8]:
# ratio = GBP.get_seeding_ratio(ipath, 1981, 2005)

fpath = gv.pre_path

### should these be changing depending on what years of data we're running CHAZ on? 
iy1 = 1981
iy2 = 2005
### from getBam -- get_seeding_ratio

f = gv.opath
bt_obs = nc.Dataset(f,'r')
NS = np.argwhere((bt_obs['StormYear'][:]>=iy1)&(bt_obs['StormYear'][:]<=iy2))[:,0].shape[0]

totalNS = np.float64(NS)/gv.survivalrate
ratio = 1.0

for iratio in range(0,1): ## I mad mistake to make it 2 for most of CMIP5 runs, and need to fix that
    ## seeding storms ####
    time1 = time.time()
    climInitLon = []
    climInitLat = []
    climInitDate = []
    for iy in range(iy1,iy2+1,1):
        tcgiFile = fpath+'/'+gv.TCGIinput+'_'+int2str(iy,4)+'.mat'
        gi = np.rollaxis(loadmat(tcgiFile)['TCGI'],2,0)
        #print iy, np.nansum(gi)
        if iy == iy1:
            xlon =loadmat(tcgiFile)['lon']
            xlat =loadmat(tcgiFile)['lat']
            gxlon,gxlat = np.meshgrid(xlon,xlat)
        climInitLon,climInitLat,climInitDate = \
            TCgiSeeding (gxlon,gxlat,gi,climInitLon,climInitLat,climInitDate,ratio,iy)
    NS = len(climInitLon)
    #print NS, totalNS,ratio
    ratio = np.float64(totalNS)/np.float64(NS)

In [9]:
ratio = np.float64(1.5260144386357979)

##ratio

In [ ]:
fpath = gv.pre_path
iy = 2009
### from getBam -- getSeeding

climInitLon = []
climInitLat = []
climInitDate = []

# Load the tropical cyclone genesis index (TCGI)
tcgiFile = fpath+gv.TCGIinput+'_'+int2str(iy,4)+'.mat'
gi = np.rollaxis(loadmat(tcgiFile)['TCGI'],2,0)

# Load longitude and latitude data
xlon =np.float(loadmat(tcgiFile)['lon'])
xlat =np.float(loadmat(tcgiFile)['lat'])

# Create a meshgrid of longitude and latitude

### get red of uint16
gxlon,gxlat = np.meshgrid(xlon,xlat)

# Generate seeds based on the genesis index
# climInitLon,climInitLat,climInitDate = \
#     TCgiSeeding (gxlon,gxlat,gi,climInitLon,climInitLat,climInitDate,ratio,iy)
# climInitDate = np.array(climInitDate).ravel()
# climInitLon = np.array(climInitLon).ravel()
# climInitLat = np.array(climInitLat).ravel()

In [11]:

# loop through year (iy) by month (im) 
for im in range(12):
    xk, pk = np.arange(gi[im,:,:].size), gi[im,:,:].ravel()

    # lon_flat = gxlon.ravel()
    # lat_flat = gxlat.ravel()
    # valid = np.isfinite(pk) & np.isfinite(lon_flat) & np.isfinite(lat_flat)
    # xk, pk = xk[valid], pk[valid]

    ####### UPDATING

    ## TODO: find way to fix when multiple runs x1 produces empty/full of nans 
    ## remove points where gi OR the coordinate grids are NaN
    # lon_flat = gxlon.ravel()
    # lat_flat = gxlat.ravel()
    # valid = np.isfinite(pk) & np.isfinite(lon_flat) & np.isfinite(lat_flat)
    # xk, pk = xk[valid], pk[valid]

    # custm = stats.rv_discrete(name='custm', values=(xk, pk/pk.sum()), seed=rng)

    # n_seeds = np.int_(np.rint(pk.sum()*ratio))
    # r = custm.rvs(size=n_seeds)

    # iix = gxlon.ravel()[r.ravel()]
    # iiy = gxlat.ravel()[r.ravel()]

    # iday = rng.choice(np.arange(calendar.monthrange(iy,im+1)[1]),
    #                    size=n_seeds, replace=True)

    # if iday.size > 0:
    #     for id in range(iday.size):

    #         lon0, lat0 = iix[id], iiy[id]

    #         ## belt-and-suspenders: skip if somehow still NaN/inf
    #         if not (np.isfinite(lon0) and np.isfinite(lat0)):
    #             continue

    #         ## linspace guarantees a fixed, non-empty length regardless
    #         ## of float rounding, unlike arange
    #         x1 = np.linspace(lon0-1, lon0+1, 201)
    #         y1 = np.linspace(lat0-1, lat0+1, 201)

    #         xx = rng.choice(x1, 1)
    #         yy = rng.choice(y1, 1)

    #         if gv.debugging: print(xx)
    #         if gv.debugging: print(yy)

    #         climInitDate.append(datetime(iy,im+1,iday[id]+1,0,0))
    #         climInitLon.append(xx)
    #         climInitLat.append(yy)

    ## OLD

    ### this maybe isn't working as well as it could 
    ## remove NaNs
    dummy = xk*pk
    xk, pk = xk[dummy==dummy], pk[dummy==dummy] # (dummy==dummy is False for NaNs)
    # Build discrete probability distribution (custm) using gi values as weights
    # xk = possible grid indices, pk/pk.sum() = normalized probabilities
    custm = stats.rv_discrete(name='custm', values=(xk, pk/pk.sum()), seed = rng)

    #print(custm)

#### REROLLING RANDOM IF there is a 0 in iix
    ## randomly select grid point from custm 
    ## number of points selected [np.rint(pk.sum()*ratio)]
        # proportional to sum of gensis probability * ratio
    #r = custm.rvs(size=np.int_(np.rint(pk.sum()*ratio)))
    ## map selected grid points into lon/lat coordinated
    #iix = gxlon.ravel()[r.ravel()]
    #iiy = gxlat.ravel()[r.ravel()]

####

    n = np.int_(np.rint(pk.sum() * ratio))

    r = custm.rvs(size=n)
    iix = gxlon.ravel()[r.ravel()]
    iiy = gxlat.ravel()[r.ravel()]

    # while np.any(iix == 0):
    #     bad = (iix == 0)
    #     # Re-roll bad positions
    #     r[bad] = custm.rvs(size=np.sum(bad))

    #     # Recalculate those positions
    #     iix[bad] = gxlon.ravel()[r[bad]]

    # while np.any(iiy == 0):
    #     bad = (iiy == 0)
    #     r[bad] = custm.rvs(size=np.sum(bad))
    #     iiy[bad] = gxlat.ravel()[r[bad]]


    ## randomly select day in month to assign each seed 
    ## update to use set rng
    iday = rng.choice(np.arange(calendar.monthrange(iy,im+1)[1]), size=np.int_(np.rint(pk.sum()*ratio)), replace=True)


    # if there are days assigned a seed in the month:
    if iday.size>0:
        for id in range(iday.size):

            ## this is occasionally generating empty arrays
            x1 = np.arange(iix[id]-1,iix[id]+1.01, 0.01)
            y1 = np.arange(iiy[id]-1,iiy[id]+1.01, 0.01)

            #print(x1)
            
            ## update to use set rng
            xx = rng.choice(x1, 1)
            yy = rng.choice(y1, 1)

            # if gv.debugging: print(xx)
            # if gv.debugging: print(yy)

            # append generated seed information to appropriate lists
            climInitDate.append(datetime(iy,im+1,iday[id]+1,0,0))
            climInitLon.append(xx)
            climInitLat.append(yy)


ValueError: a cannot be empty unless no samples are taken

In [ ]:
pk.min()

np.float64(0.0)

In [31]:
gxlon


array([[  0,   2,   4, ..., 354, 356, 358],
       [  0,   2,   4, ..., 354, 356, 358],
       [  0,   2,   4, ..., 354, 356, 358],
       ...,
       [  0,   2,   4, ..., 354, 356, 358],
       [  0,   2,   4, ..., 354, 356, 358],
       [  0,   2,   4, ..., 354, 356, 358]], shape=(91, 180), dtype=uint16)

In [13]:
len(r)

8

In [17]:
iix

array([156, 176, 192, 106,  82,  84,   0, 176], dtype=uint16)

In [27]:
np.int_(iix[id])-1

np.int64(-1)

In [29]:
np.arange(np.int_(iix[id])-1,np.int_(iix[id])+1.01, 0.01)

array([-1.0000000e+00, -9.9000000e-01, -9.8000000e-01, -9.7000000e-01,
       -9.6000000e-01, -9.5000000e-01, -9.4000000e-01, -9.3000000e-01,
       -9.2000000e-01, -9.1000000e-01, -9.0000000e-01, -8.9000000e-01,
       -8.8000000e-01, -8.7000000e-01, -8.6000000e-01, -8.5000000e-01,
       -8.4000000e-01, -8.3000000e-01, -8.2000000e-01, -8.1000000e-01,
       -8.0000000e-01, -7.9000000e-01, -7.8000000e-01, -7.7000000e-01,
       -7.6000000e-01, -7.5000000e-01, -7.4000000e-01, -7.3000000e-01,
       -7.2000000e-01, -7.1000000e-01, -7.0000000e-01, -6.9000000e-01,
       -6.8000000e-01, -6.7000000e-01, -6.6000000e-01, -6.5000000e-01,
       -6.4000000e-01, -6.3000000e-01, -6.2000000e-01, -6.1000000e-01,
       -6.0000000e-01, -5.9000000e-01, -5.8000000e-01, -5.7000000e-01,
       -5.6000000e-01, -5.5000000e-01, -5.4000000e-01, -5.3000000e-01,
       -5.2000000e-01, -5.1000000e-01, -5.0000000e-01, -4.9000000e-01,
       -4.8000000e-01, -4.7000000e-01, -4.6000000e-01, -4.5000000e-01,
      

In [ ]:
iday

array([12,  9,  9,  7,  0, 17,  7, 26])

In [ ]:
iix

array([ 52, 128, 130, 136, 154,  62,  56,  52], dtype=uint16)

In [ ]:
id

7

In [ ]:
iix[id]

np.uint16(52)

In [ ]:
iix[id]

np.uint16(52)